# 5.8 - Deep learning: sequence models of returns, and a neural ensemble on the features

> **Needs:** notebook 5.2's features and the folds of 5.3; the optional `deep` dependency group (torch); the database.
>
> **Produces:** walk-forward predictions in the artifact store and the registered trials.

Two kinds of neural network, trained on the processor (CPU): when this notebook was run the machine had no
usable graphics card (GPU), so every network is small and trained on sampled data. A GPU has worked since
November 2026 and the models can use it (`device=`); these results are the CPU's:

1. **Sequence models of weekly returns** (*Machine Learning for Trading*, chapter 13). Each stock's last 52 weekly
   returns - each week as its rank among the members, so the market's move is taken out - are read by a network that
   predicts next week's return over the average member. Three architectures: **NLinear** (one linear layer: the
   baseline that many deep forecasters fail to beat), an **LSTM** (a recurrent network with a memory) and **TSMixer**
   (mixing layers across the weeks). The yardstick is the simplest weekly signal: last week's **reversal**.
2. **An ensemble of multilayer perceptrons on the 45 features of 5.2**, in the spirit of TabM (Gorishniy et al., 2024),
   on exactly the monthly inputs, target and folds of 5.3 and 5.4 - so it is judged against the linear models,
   gradient boosting and the composite.

Everything is out of sample: refitted each January on earlier years only, judged on 2015-2024; seeded and
deterministic, so a rerun repeats it; every architecture registered in the new family `ml4t_deep`.

## How these networks work

A **neural network** is a stack of simple functions: each layer multiplies its input by a matrix of weights, adds a
bias and (in hidden layers) applies a non-linear function such as the ReLU, $\max(0, z)$. The weights are fitted by
**gradient descent** on the squared error, in small random batches (here with the Adam optimiser), and training stops
when the error on held-out data stops falling (**early stopping**), keeping the best weights.

* **NLinear** (Zeng et al., 2023) - $\hat y = w^\top (s - s_{52}) + s_{52}$: a weighted sum of the sequence $s$ after
  subtracting its last value. No hidden layer at all.
* **LSTM** (Hochreiter and Schmidhuber, 1997) - reads the 52 weeks in order, updating a 32-number *cell state* through
  gates that decide what to keep, forget and output; the final state predicts the target.
* **TSMixer** (Chen et al., 2023) - two blocks, each adding to the sequence a small network's transformation of it
  (a *residual* connection), after normalising it.
* **MLP ensemble** - eight networks with two hidden layers of 64 ReLU units and dropout (10 % of units switched off at
  random in training, against overfitting), each from a different random start, their predictions averaged.

Returns are tiny numbers, so each network learns the target divided by its standard deviation; a ranking is
unaffected.

In [1]:
import pandas as pd
from IPython.display import display

from equity_research.data import artifacts as store
from equity_research.data.cache import ResearchCache
from equity_research.data.universe import load_universe_data
from equity_research.ml import boosting as gb
from equity_research.ml import deep as dp
from equity_research.ml import linear as ln
from equity_research.ml.features import unstack
from research_core import charts
from research_core.analytics.signals import evaluation as ev
from research_core.analytics.signals import labels as lb
from research_core.analytics.signals import validation as va
from research_core.data.db.engine import get_engine
from research_core.engine.registry import record_search, trial_count

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
charts.style()

UNIVERSE, START, END = "SP1500", "2005-01-01", "2024-12-31"  # END: the last day before the hold-out
FIRST_TEST, LAST_TEST, TRAIN_YEARS, OPTUNA_TRIALS = 2015, 2024, 10, 40
cache = ResearchCache()
SNAPSHOT = store.data_snapshot(cache)
data = load_universe_data(cache, UNIVERSE, START, END)
dates = data.rebalances
test_dates = dates[(dates.year >= FIRST_TEST) & (dates.year <= LAST_TEST)]
FEATURES = {"universe": UNIVERSE, "start": START, "end": END, "dates": "month ends", "version": 1}
long = store.load("features", "company_month_end", FEATURES, snapshot=SNAPSHOT)
assert long is not None, "run notebooks 5.2-5.4 first"
monthly = store.cached(
    "labels",
    "fwd_ret_month",
    {"universe": UNIVERSE, "start": START, "end": END, "dates": "rebalances"},
    lambda: lb.period_returns(data.returns, dates),
    snapshot=SNAPSHOT,
)
x = ln.centred_ranks(long)
y = lb.trimmed_excess(monthly, data.members)
ends = va.next_date_ends(dates)
folds = va.walk_forward(dates, ends, train_years=TRAIN_YEARS, first_test=FIRST_TEST, last_test=LAST_TEST)
STUDY = FEATURES | {
    "target": "trimmed excess, 1 %",
    "train_years": TRAIN_YEARS,
    "first_test": FIRST_TEST,
    "flat": "ic 0",
}

left out 1078 facts of 2 companies in ['GBP']


## Step 1 - the weekly sequence models

Week ends from 2005 to 2024; each week's return as a centred rank among the members; the target is next week's
return over the average member (extreme 1 % clipped). Each model is refitted each January on up to 150,000 sequences
sampled from every earlier year, validated on the last training year; results kept in the artifact store.

In [2]:
weeks = dp.week_ends(pd.DatetimeIndex(data.returns.index))
past = dp.weekly_ranks(data.returns, data.members, weeks)
week_target = lb.trimmed_excess(lb.period_returns(data.returns, weeks), data.members)
SEQ = {
    "universe": UNIVERSE,
    "start": START,
    "end": END,
    "lookback": dp.LOOKBACK,
    "max_rows": dp.MAX_ROWS,
    "epochs": 10,
    "patience": 2,
    "first_test": FIRST_TEST,
}


def sequence_scores(name: str) -> pd.DataFrame:
    """One sequence model's out-of-sample weekly scores."""
    run = dp.walk_forward_sequences(
        name, past, week_target, data.members, first_test=FIRST_TEST, last_test=LAST_TEST, epochs=10, patience=2
    )
    return run.scores


weekly = {
    name: store.cached(
        "deep", f"weekly-{name}", SEQ | {"model": name}, lambda n=name: sequence_scores(n), snapshot=SNAPSHOT
    )
    for name in dp.SEQUENCE_MODELS
}
test_weeks = weeks[(weeks.year >= FIRST_TEST) & (weeks.year <= LAST_TEST)]
weekly["reversal (minus last week's rank)"] = -past.loc[test_weeks]
weekly_ics = pd.DataFrame(
    {n: ev.ic_series(s.loc[s.index.isin(test_weeks)], week_target, data.members) for n, s in weekly.items()}
).dropna(how="all")
weekly_table = ev.ic_summary(weekly_ics, lags=0)[["mean", "t_nw", "positive", "dates"]]
display(weekly_table.style.format({"mean": "{:+.4f}", "t_nw": "{:+.2f}", "positive": "{:.0%}", "dates": "{:.0f}"}))

,mean,t_nw,positive,dates
nlinear,+0.0121,+2.12,55%,522
lstm,+0.0071,+1.03,50%,522
tsmixer,+0.0077,+1.74,53%,522
reversal (minus last week's rank),+0.0135,+2.03,51%,522


## Step 2 - the MLP ensemble on the features, against everything before it

The same inputs, target and folds as 5.3 and 5.4, judged by the monthly rank IC against next month's return,
2015-2024, beside the linear models, gradient boosting and the composite (read back from the artifact store).

In [3]:
MLP = STUDY | {"members": dp.MEMBERS, "epochs": 20, "patience": 3}
mlp = store.cached(
    "deep",
    "mlp-ensemble",
    MLP,
    lambda: dp.walk_forward_mlp(x, y, folds, epochs=20, patience=3).scores,
    snapshot=SNAPSHOT,
)
panels = unstack(long)
linear_runs = {
    n: ln.stored_walk_forward(x, y, folds, ends, m, name=n, config=STUDY, snapshot=SNAPSHOT)
    for n, m in ln.MODELS.items()
}
grid_run = gb.stored_walk_forward(x, y, folds, ends, gb.grid(), name="grid", config=STUDY, snapshot=SNAPSHOT)
loss = gb.grid()[str(grid_run.tuning.loc[min(grid_run.tuning.index)].idxmax())].loss
_, best = gb.stored_search(
    x, y, folds[0].train, ends, loss=loss, n_trials=OPTUNA_TRIALS, config=STUDY, snapshot=SNAPSHOT
)
tuned_run = gb.stored_walk_forward(x, y, folds, ends, {"optuna": best}, name="optuna", config=STUDY, snapshot=SNAPSHOT)
scores = (
    {"MLP ensemble": mlp}
    | {ln.MODELS[n].label: r.scores for n, r in linear_runs.items()}
    | {
        "GBM, grid": grid_run.scores,
        "GBM, Optuna": tuned_run.scores,
        "Composite (issuance + ROCE)": ln.composite(
            {"issuance": -panels["share_change"], "ROCE": panels["ROCE"]}, data.members
        ).loc[test_dates],
    }
)
ics = pd.DataFrame({n: ev.ic_series(s, monthly, data.members) for n, s in scores.items()}).loc[test_dates]
table = ev.ic_summary(ics, lags=0)[["mean", "t_nw", "positive"]]
table["turnover"] = [ev.top_turnover(s.loc[test_dates], data.members).mean() for s in scores.values()]
display(
    table.sort_values("mean", ascending=False).style.format(
        {"mean": "{:+.4f}", "t_nw": "{:+.2f}", "positive": "{:.0%}", "turnover": "{:.0%}"}
    )
)
versus = pd.DataFrame({other: ln.ic_difference(ics["MLP ensemble"], ics[other]) for other in ics.columns[1:]}).T
print("The MLP ensemble's monthly IC less each other score's")
display(versus.style.format({"mean difference": "{:+.4f}", "t_nw": "{:+.2f}", "dates": "{:.0f}"}))

,mean,t_nw,positive,turnover
"GBM, grid",+0.0276,+2.75,58%,53%
"GBM, Optuna",+0.0274,+2.69,59%,53%
Composite (issuance + ROCE),+0.0252,+3.21,66%,8%
Logistic regression,+0.0245,+2.81,66%,35%
MLP ensemble,+0.0227,+2.34,63%,43%
Ridge,+0.0150,+1.81,65%,37%
Elastic net,+0.0090,+0.94,60%,34%
Lasso,+0.0057,+0.62,56%,32%


The MLP ensemble's monthly IC less each other score's


,mean difference,t_nw,dates
Ridge,+0.0077,+1.84,119
Lasso,+0.0170,+2.82,119
Elastic net,+0.0137,+2.58,119
Logistic regression,-0.0019,-0.20,119
"GBM, grid",-0.0049,-0.39,119
"GBM, Optuna",-0.0047,-0.37,119
Composite (issuance + ROCE),-0.0025,-0.23,119


## Step 3 - registering every architecture

In [4]:
# the registry is the shared database: connected only now, as a serverless database pauses when idle
engine = get_engine()
points = [
    (
        SEQ | {"model": name},
        {"oos_ic": float(weekly_table.loc[name, "mean"]), "oos_t": float(weekly_table.loc[name, "t_nw"])},
    )
    for name in dp.SEQUENCE_MODELS
]
points.append(
    (
        MLP | {"model": "mlp ensemble"},
        {"oos_ic": float(table.loc["MLP ensemble", "mean"]), "oos_t": float(table.loc["MLP ensemble", "t_nw"])},
    )
)
record_search(engine, dp.FAMILY, points, asset_class="equity", search="architectures")
print("Trials registered in the family:", trial_count(engine, dp.FAMILY))

Trials registered in the family: 4


## Findings

*Rerun on 17 November 2026 on quant_research_local (history from 2000). The weekly networks' ICs rose - NLinear's
from +0.0071 to +0.0121 - and the order of the LSTM and TSMixer swapped; reversal still ranks best. One seed per
network, so a rerun on new data moves them this much.*

**In short: the deep sequence models do not beat last week's reversal, and the neural ensemble on the features joins
the group of models that rank about as well as the plain composite, and no better.**

**1. Weekly sequences: a one-line rule wins (step 1).** Mean weekly rank IC against next week's return, 522 weeks,
2015-2024:

| Model | mean IC | t | weeks positive |
|---|---|---|---|
| Reversal (minus last week's rank) | +0.0135 | +2.03 | 51 % |
| NLinear | +0.0121 | +2.12 | 55 % |
| TSMixer | +0.0077 | +1.74 | 53 % |
| LSTM | +0.0071 | +1.03 | 50 % |

Given a year of weekly history for every stock, none of the three networks found more than the simplest pattern
in it: last week's winners give some back. All three found less, and the one closest to reversal is the linear one -
which fits what Zeng et al. (2023) found for many forecasting tasks, that a single linear layer often matches far
more complex networks: on noisy series, more capacity fits more noise. (In the run of 14 November the order below it
was LSTM then TSMixer; with one seed each, the two are not distinguishable.) And even reversal's +0.0135 is a weekly
signal, which notebook 5.1 found costly to trade (a week's typical move covers about five round trips). Sizing the
networks for a CPU - one seed, 150,000 sampled sequences a year, early stopping - keeps them modest; larger ones are a
guess this evidence does not support.

**2. The MLP ensemble on the features (step 2).** Monthly, out of sample, against everything before it:

| Score | mean IC | t | turnover |
|---|---|---|---|
| GBM, grid (5.4) | +0.0276 | +2.75 | 53 % |
| GBM, Optuna (5.4) | +0.0274 | +2.69 | 53 % |
| Composite (issuance + ROCE) | +0.0252 | +3.21 | 8 % |
| Logistic regression (5.3) | +0.0245 | +2.81 | 35 % |
| **MLP ensemble** | **+0.0227** | **+2.34** | 43 % |
| Ridge (5.3) | +0.0150 | +1.81 | 37 % |

* **Against the penalised regressions** - the ensemble's monthly IC is significantly above the lasso's (+0.0170, t
  2.82) and the elastic net's (+0.0137, t 2.58), and above ridge's by a margin just short of significance (+0.0077,
  t 1.84). Some non-linearity helps over a straight line fitted to returns.
* **Against the rest** - it is not distinguishable from logistic regression (-0.0019, t -0.20), gradient boosting
  (-0.0049 and -0.0047, |t| below 0.4) or the composite (-0.0025, t -0.23).

**3. The pattern across 5.3, 5.4 and this notebook.** The models fall into two groups:

* **Fitting raw returns with straight lines** - ridge, lasso and the elastic net: ICs of +0.006 to +0.015.
* **Bending, or caring only about direction** - gradient boosting, the MLP ensemble and logistic regression: ICs of
  +0.023 to +0.028, about the composite's +0.025.

Within the second group no model is distinguishable from another, or from the composite, which needs no fitting and
replaces 8 % of its top fifth a month against their 35-53 %. Five architectures and over eighty registered configurations later, the features of 5.2
hold an IC of about 0.02-0.03 a month, and no model extracts more that the data can confirm.

The four architectures here are registered in `ml4t_deep`.

## Glossary

* **Neural network** - layers of weighted sums and simple non-linear functions, fitted by gradient descent.
* **Recurrent network (LSTM)** - a network that reads a sequence step by step and carries a memory between steps.
* **Residual connection** - adding a layer's input to its output, so the layer learns a correction.
* **Dropout** - switching off a random share of units during training, so no unit can be relied on alone.
* **Early stopping** - ending training when the error on held-out data stops improving.
* **Ensemble** - the average of several models' predictions; it varies less than any one of them.
* **Short-term reversal** - the tendency of last week's (or month's) winners to give some of it back.